<img src="https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aureliensalas/python_finance/blob/main/ml_crypto/cours/seance1_cours.ipynb)

# Séance 7 : expliquer, puis prévoir

**Cours** · 2h · machine learning, première partie

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/aureliensalas/python_finance/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire ce qu'est l'apprentissage automatique, et en reconnaître les trois familles
- mettre des données de côté pour noter un modèle honnêtement
- expliquer ce que fait une régression linéaire, sans formule : la droite des moindres carrés
- mesurer un modèle avec trois nombres, R², MAE et RMSE, et savoir lequel regarder
- distinguer **expliquer** et **prévoir**
- reconnaître un modèle qui a appris par cœur

## La phrase du bloc

> **Un modèle n'a pas de valeur parce qu'il colle aux données. Il en a s'il tient sur des données qu'il n'a jamais vues.**

Le bloc 3 demandait : *ce que je vois est-il réel ?* Le bloc 4 demande : *ce que j'ai trouvé tient-il ailleurs ?*

> **Les cellules « Prédire ».** Avant de les exécuter, on annonce **à voix haute** la valeur attendue. Rien à écrire : on dit, on exécute, on compare.

Exécutez d'abord la cellule de setup.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pd.set_option("display.max_rows", 12)
BASE = "https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/pandas_crypto/data/"                # crypto.csv, du bloc 2
BASE_STATS = "https://cdn.jsdelivr.net/gh/aureliensalas/python_finance@main/stats_crypto/data/"    # rendements.csv, du bloc 3


def verifier(nom, condition, indice=""):
    if condition:
        print("OK       -", nom)
    else:
        print("A REVOIR -", nom, ":", indice)

La cellule suivante contient deux fonctions de dessin : `nuage`, qui trace un nuage de points avec des droites par-dessus, et `barres`, qui compare quelques nombres. **Vous n'avez pas à la lire** : exécutez-la, puis oubliez-la.

In [ ]:
COULEURS = ["#B4341F", "#2E7D5B", "#B26B00", "#6B3FA0", "#1F2933"]


def nuage(x, y, courbes=None, titre="", xlabel="", ylabel=""):
    """Le nuage des points (x, y), et par-dessus une ou plusieurs courbes.

    courbes : {"nom affiché": valeurs}, où valeurs est un nombre (une ligne horizontale)
    ou une colonne de même longueur que x (une valeur par point).
    """
    x = pd.Series(np.asarray(x, dtype=float).ravel())
    y = pd.Series(np.asarray(y, dtype=float).ravel())
    ordre = np.argsort(x.values)
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    ax.scatter(x, y, s=9, alpha=0.3, color="#1F4E79", edgecolor="none")
    for k, (nom, valeurs) in enumerate((courbes or {}).items()):
        v = np.full(len(x), float(valeurs)) if np.ndim(valeurs) == 0 else np.asarray(valeurs, dtype=float).ravel()
        ax.plot(x.values[ordre], v[ordre], lw=2.4, color=COULEURS[k % len(COULEURS)], label=nom)
    if courbes:
        ax.legend(loc="upper left", frameon=False)
    ax.axhline(0, color="#9AA5B1", lw=0.6, zorder=0)
    ax.axvline(0, color="#9AA5B1", lw=0.6, zorder=0)
    for cote in ("top", "right"):
        ax.spines[cote].set_visible(False)
    ax.set_title(titre, loc="left", fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.show()


def barres(valeurs, titre="", unite=""):
    """Des barres horizontales, une par nom : {"nom": nombre}. La valeur est écrite au bout."""
    noms = list(valeurs)[::-1]
    nombres = [float(valeurs[n]) for n in noms]
    fig, ax = plt.subplots(figsize=(7.5, 0.6 + 0.55 * len(noms)))
    ax.barh(noms, nombres, color=["#1F4E79" if n >= 0 else "#B4341F" for n in nombres], height=0.55)
    haut = max(abs(n) for n in nombres)
    for i, n in enumerate(nombres):
        texte = f"{n:,.0f}".replace(",", " ") if abs(n) >= 100 else f"{n:.3g}".replace(".", ",")
        ax.text(n + (0.01 * haut if n >= 0 else -0.01 * haut), i, f" {texte} {unite} ",
                va="center", ha="left" if n >= 0 else "right", fontsize=10)
    ax.axvline(0, color="#1F2933", lw=0.8)
    for cote in ("top", "right"):
        ax.spines[cote].set_visible(False)
    ax.set_xlim(min(0, min(nombres)) * 1.25, max(0, max(nombres)) * 1.25)
    ax.set_title(titre, loc="left", fontweight="bold")
    plt.show()

## 0. Qu'est-ce qu'apprendre ?

Pas de définition pour commencer. Six situations, et une consigne : **rangez-les en trois tas**, à l'oral, avec votre voisin. On ne vous dit pas encore ce que sont les tas.

| | Situation |
|---|---|
| A | Une banque a 200 000 dossiers de crédit passés, chacun marqué « remboursé » ou « défaut ». Elle veut évaluer un nouveau dossier. |
| B | Un assureur a l'historique de ses sinistres automobiles, avec le montant payé. Il veut estimer le coût d'un nouveau sinistre. |
| C | Un gérant a les profils de 50 000 clients, sans aucune étiquette. Il veut les regrouper en quelques familles. |
| D | Un algorithme place des ordres sur un marché, encaisse des gains et des pertes, et corrige sa stratégie au fil des résultats. |
| E | Une plateforme a des millions de paiements par carte, dont quelques centaines de fraudes avérées. Elle veut repérer les prochaines. |
| F | Sept cryptomonnaies sur huit ans. On cherche lesquelles bougent ensemble, sans idée de départ. |

Qu'est-ce qui réunit A, B et E ? Qu'est-ce qui réunit C et F ? Pourquoi D est-il à part ?

<details><summary><b>Les trois tas, et leurs noms</b> (cliquez après avoir rangé)</summary>

- **A, B, E** : on a les **réponses** du passé — remboursé ou non, le montant payé, fraude ou non — et on apprend à les reproduire sur des cas nouveaux. C'est l'apprentissage **supervisé**.
- **C, F** : il n'y a pas de réponse. On cherche une structure dans les données. C'est l'apprentissage **non supervisé**.
- **D** : ni réponse ni structure, mais des essais et une récompense. C'est l'apprentissage **par renforcement**.

Et dans le supervisé, deux cas selon ce qu'on cherche :

| on prédit... | exemple | ça s'appelle |
|---|---|---|
| un **nombre** | B, le coût du sinistre | une **régression** |
| une **catégorie** | A et E, oui ou non | une **classification** |

Aujourd'hui, la régression. La prochaine fois, la classification. Le non supervisé et le renforcement sont nommés, et on les laisse là.
</details>

## 1. L'examen dont on a le corrigé

Avant de construire le moindre modèle, une question : **quand on en aura un, comment saura-t-on s'il est bon ?**

Vous connaissez la réponse. Vous la vivez chaque semestre.

| Dans ce cours | En machine learning |
|---|---|
| les problem sets, avec leur corrigé | les données d'**entraînement** |
| l'examen, dont vous ne voyez ni les questions ni les réponses avant | les données de **test** |
| ce que je veux savoir : savez-vous faire un exercice que vous n'avez jamais vu ? | ce qu'on veut savoir : le modèle marche-t-il sur des données qu'il n'a jamais vues ? |

**Si je mettais à l'examen exactement les exercices du problem set, qu'est-ce que la note mesurerait ?** Votre mémoire, pas votre compréhension. Un étudiant qui aurait appris les corrigés par cœur aurait 20, et serait perdu devant le premier exercice nouveau.

Un modèle noté sur les données qui ont servi à le construire, c'est exactement ça : **il a vu les réponses**. Sa note ne vaut rien.

D'où la règle : **avant de commencer**, on met de côté une partie des données. On n'y touche pas pendant l'apprentissage. On ne la sort qu'à la fin, pour noter.

### Les données

`rendements.csv`, le fichier large de la séance 6 : une ligne par jour, une colonne par monnaie. On garde les jours où les sept sont cotées — SOL n'arrive qu'en avril 2020.

In [ ]:
rendements = pd.read_csv(BASE_STATS + "rendements.csv")
rendements = rendements.dropna().reset_index(drop=True)
rendements.head()

On va chercher à relier ETH à BTC. Deux objets, et la différence de crochets compte :

- `X`, **ce qu'on donne** au modèle : une **table**, avec deux crochets, parce qu'on pourra lui donner plusieurs colonnes ;
- `y`, **ce qu'on cherche** : une seule **colonne**, un seul crochet.

In [ ]:
X = rendements[["BTC"]]
y = rendements["ETH"]
print(len(X), len(y))

### La fonction qui met de côté

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
```

- **Quatre sorties, dans cet ordre** : ce qu'on donne puis ce qu'on cherche, l'entraînement puis le test. Toujours le même ordre.
- `test_size=0.2` : un cinquième des jours mis de côté.
- `shuffle=False` : **ne pas mélanger**. Par défaut, la fonction tire les jours au hasard. Sur des dates, c'est interdit : le modèle apprendrait sur le 4 janvier pour être noté sur le 3. L'examen porterait sur une semaine dont il a déjà vu la suite. Avec `shuffle=False`, on coupe dans l'ordre : **on apprend sur le passé, on est noté sur le futur**, comme dans la vraie vie.

### Écrire

Coupez `X` et `y` avec cette ligne.

In [ ]:
verifier("la taille de l'entraînement", len(X_train) == 1867 and len(y_train) == 1867, "test_size=0.2")
verifier("la taille du test", len(X_test) == 467, "test_size=0.2")
verifier("l'ordre des dates", X_test.index.min() > X_train.index.max(), "shuffle=False : sinon les jours sont mélangés")

### Prédire

In [ ]:
print(len(X_train) + len(X_test) == len(rendements))

In [ ]:
print(rendements["date"][1866], rendements["date"][1867])

On apprend jusqu'au 21 mai 2025. On est noté à partir du 22. **À partir de maintenant, `X_test` et `y_test` sont sous enveloppe scellée** : on ne les ouvre qu'à la section 3.

## 2. La régression linéaire

### Le nuage

Chaque point est une journée d'entraînement : en abscisse, ce qu'a fait BTC ; en ordonnée, ce qu'a fait ETH le même jour.

In [ ]:
nuage(X_train["BTC"], y_train, titre="Une journée = un point", xlabel="BTC (%)", ylabel="ETH (%)")

Un nuage allongé le long d'une diagonale : quand BTC monte, ETH monte, et à peu près d'autant. C'est la corrélation de 0,82 de la séance 6, vue en image.

On cherche **la** droite qui passe au mieux dans ce nuage. **Au mieux selon quoi ?** Il faut un nombre pour noter une droite.

### D'abord, le modèle bête

La droite la plus simple qui soit : **horizontale**, à la hauteur de la moyenne d'ETH. Elle ignore complètement BTC et répond toujours la même chose. C'est notre point de comparaison pour tout le bloc.

In [ ]:
moyenne = y_train.mean()
print(moyenne)
nuage(X_train["BTC"], y_train, {"modèle bête : toujours la moyenne": moyenne}, titre="Le modèle bête", xlabel="BTC (%)", ylabel="ETH (%)")

### Noter une droite : la somme des erreurs au carré

Pour chaque jour, **l'erreur** : l'écart vertical entre le point et la droite. On l'élève **au carré** — sinon les erreurs au-dessus et en dessous s'annuleraient — et on additionne tout.

In [ ]:
erreurs = y_train - moyenne
print((erreurs ** 2).sum())

**32 426.** C'est la note du modèle bête. Plus elle est petite, plus la droite colle au nuage.

### Prédire

Laquelle de ces trois droites aura la plus petite somme ? `y = 0,5 x`, `y = x`, `y = 1,5 x`.

In [ ]:
nuage(X_train["BTC"], y_train,
      {"y = 0,5 x": 0.5 * X_train["BTC"], "y = x": X_train["BTC"], "y = 1,5 x": 1.5 * X_train["BTC"]},
      titre="Trois droites candidates", xlabel="BTC (%)", ylabel="ETH (%)")

### Écrire

La somme des erreurs au carré pour chacune des trois droites : `sce_05`, `sce_1`, `sce_15`. Même calcul que pour le modèle bête ; seule change la prévision, qui vaut `0.5 * X_train["BTC"]` au lieu de `moyenne`.

In [ ]:
verifier("sce_05", abs(sce_05 - ((y_train - 0.5 * X_train["BTC"]) ** 2).sum()) < 1, "((y_train - 0.5 * X_train['BTC']) ** 2).sum()")
verifier("sce_1", abs(sce_1 - ((y_train - X_train["BTC"]) ** 2).sum()) < 1, "la prévision de y = x, c'est X_train['BTC'] tout court")
verifier("sce_15", abs(sce_15 - ((y_train - 1.5 * X_train["BTC"]) ** 2).sum()) < 1, "même calcul avec 1.5")

| droite | somme des erreurs au carré |
|---|---|
| modèle bête : la moyenne | 32 426 |
| `y = 0,5 x` | 17 487 |
| `y = 1,5 x` | 15 383 |
| `y = x` | **11 772** |

Les trois font mieux que le modèle bête : **BTC apporte de l'information sur ETH**. La diagonale gagne. Reste à trouver, parmi **toutes** les droites possibles, celle qui rend la somme la plus petite.

### `fit` fait exactement ça

In [ ]:
modele = LinearRegression().fit(X_train, y_train)
print(modele.coef_[0], modele.intercept_)

Deux nombres : la **pente**, 1,056, et l'**ordonnée à l'origine**, 0,026. La droite trouvée est `y = 1,056 x + 0,026`.

`coef_` est une liste, parce qu'il y a un coefficient par colonne de `X` : ici une seule, d'où le `[0]`.

### Écrire

`sce_fit`, la somme des erreurs au carré de cette droite. La prévision du modèle sur l'entraînement s'obtient avec `modele.predict(X_train)`.

In [ ]:
verifier("sce_fit", abs(sce_fit - ((y_train - modele.predict(X_train)) ** 2).sum()) < 1, "modele.predict(X_train) remplace 0.5 * X_train['BTC']")
verifier("c'est le minimum", sce_fit < sce_1, "la droite de fit doit battre y = x")

**11 711**, un peu sous `y = x`. Aucune droite ne fait mieux : c'est la définition même de ce que `fit` a cherché.

> **Une régression linéaire, c'est la droite qui rend la somme des erreurs au carré la plus petite possible.** On l'appelle la droite des **moindres carrés**. `fit` fait la recherche ; la définition, c'est vous qui venez de la construire.

In [ ]:
nuage(X_train["BTC"], y_train, {"modèle bête": moyenne, "la droite des moindres carrés": modele.predict(X_train)},
      titre="La droite trouvée par fit", xlabel="BTC (%)", ylabel="ETH (%)")

### `predict` : lire la droite

On donne un BTC, on lit l'ETH correspondant sur la droite. Il faut lui passer une **table**, avec le même nom de colonne qu'à l'entraînement.

In [ ]:
print(modele.predict(pd.DataFrame({"BTC": [2.0]})))

### Prédire

In [ ]:
print(modele.predict(pd.DataFrame({"BTC": [0.0]})))

In [ ]:
print(modele.predict(pd.DataFrame({"BTC": [-10.0]})))

## 3. Évaluer : R², puis MAE et RMSE

On ouvre l'enveloppe. Le modèle fait ses prévisions sur les 467 jours de test, qu'il n'a jamais vus.

In [ ]:
prevu = modele.predict(X_test)

### Le R², en image

Même idée qu'à la section 2, sur le test : la somme des erreurs au carré du modèle bête (la moyenne de ces journées), et celle de la droite.

### Écrire

`sce_bete_test` : la somme des carrés des écarts entre `y_test` et **sa** moyenne. `sce_modele_test` : entre `y_test` et `prevu`.

In [ ]:
verifier("sce_bete_test", abs(sce_bete_test - ((y_test - y_test.mean()) ** 2).sum()) < 1, "la moyenne de y_test, pas celle de y_train")
verifier("sce_modele_test", abs(sce_modele_test - ((y_test - prevu) ** 2).sum()) < 1, "(y_test - prevu) ** 2, puis sum()")

In [ ]:
barres({"modèle bête (la moyenne)": sce_bete_test, "la droite": sce_modele_test},
       titre="Somme des erreurs au carré, sur le test")

La barre du bas, c'est la barre du haut **moins ce que la droite a fait disparaître**. Quelle part a-t-elle fait disparaître ?

In [ ]:
r2 = 1 - sce_modele_test / sce_bete_test
print(r2)
print(r2_score(y_test, prevu))

**0,701 : la droite fait disparaître 70 % de l'erreur du modèle bête.** C'est ça, le **R²**. `r2_score` le calcule en une ligne, et vous savez maintenant d'où sort ce nombre.

| R² | ce que ça veut dire |
|---|---|
| 0 | pas mieux que la moyenne |
| 1 | plus aucune erreur |
| **négatif** | la barre du bas est plus longue que celle du haut : **pire que la moyenne** |

Le cas négatif n'est pas théorique. Vous le verrez dans vingt minutes.

### Le défaut du R² : il n'a pas d'unité

70 % de quoi ? Un gérant veut savoir **de combien** on se trompe, en points de rendement. Deux mesures répondent.

- **MAE**, l'erreur absolue moyenne : la moyenne des écarts, sans signe. `mean_absolute_error(y_test, prevu)`.
- **RMSE**, la racine de l'erreur quadratique moyenne : on moyenne les carrés, puis on prend la racine pour revenir à l'unité. `mean_squared_error(y_test, prevu) ** 0.5`.

### Écrire

`mae` et `rmse`, sur le test.

In [ ]:
verifier("mae", abs(mae - mean_absolute_error(y_test, prevu)) < 1e-9, "mean_absolute_error(y_test, prevu)")
verifier("rmse", abs(rmse - mean_squared_error(y_test, prevu) ** 0.5) < 1e-9, "la racine : ** 0.5 après mean_squared_error")

**MAE : 1,28.** En moyenne, la droite se trompe de 1,28 point sur le rendement d'ETH. C'est la phrase qu'on dit à un gérant.

**RMSE : 1,88.** Toujours au moins la MAE, parce que le carré **gonfle les grosses erreurs** : une erreur de 10 points compte 10 dans la MAE, 100 dans le RMSE avant la racine.

Ce qui compte, c'est **le rapport des deux**.

In [ ]:
print(rmse / mae)
print((y_test - prevu).abs().max())

**1,46.** Si toutes les erreurs se valaient, le rapport vaudrait 1. Plus il monte, plus **quelques journées dominent** l'erreur. Ici, la pire journée du test est à 10,2 points : les queues épaisses de la séance 5.

| Vous voulez... | Vous regardez |
|---|---|
| dire à quelqu'un « on se trompe de tant » | la **MAE**, dans l'unité de la cible |
| savoir si quelques grosses erreurs se cachent | le **RMSE**, et son rapport à la MAE |
| comparer deux modèles, deux cibles, deux périodes | le **R²**, sans unité |

## 4. Ce qu'on vient de faire : expliquer, pas prévoir

### Le coefficient a un nom : le bêta

`modele.coef_[0]` vaut 1,056. **Quand BTC bouge de 1 %, ETH bouge de 1,06 % le même jour.** C'est le **bêta** d'ETH par rapport au marché, le nombre le plus utilisé de la gestion de portefeuille.

### Et il a une p-value

L'outil de la séance 6. `LinearRegression` ne la donne pas ; `stats.linregress` la donne, en une ligne, avec la pente et son **erreur-type** — l'épaisseur de la pente.

In [ ]:
resultat = stats.linregress(X_train["BTC"], y_train)
print(resultat.slope, resultat.stderr, resultat.pvalue)

Même pente, 1,056. Une erreur-type de 0,018 : à deux erreurs-types de chaque côté, le bêta est entre **1,02 et 1,09**. Et une p-value **indiscernable de zéro** : ce n'est pas un hasard de l'échantillon.

### Prédire

Le R² de l'entraînement, et le carré de la corrélation de la séance 6. Même nombre, ou pas ?

In [ ]:
print(r2_score(y_train, modele.predict(X_train)))

In [ ]:
print(X_train["BTC"].corr(y_train) ** 2)

**Le même.** Pour une droite à une seule variable, le R² est exactement le carré de la corrélation.

### Et maintenant, le point important de la séance

**Rendez-vous bien compte de ce qu'on a fait.** On a pris BTC d'**aujourd'hui** pour « trouver » ETH d'**aujourd'hui**. Mais le jour où l'on connaît le rendement de BTC, la journée est finie : celui d'ETH, on le connaît aussi.

**On n'a rien prédit. On a expliqué.** On a mis un chiffre — 1,06, avec sa p-value — sur la relation dont on parlait à la séance 6, la corrélation. C'est utile : c'est ce qui permet de couvrir un portefeuille, on y reviendra dans le travail en autonomie. Mais ce n'est pas une prévision.

> **Expliquer une relation, c'est le métier des statistiques. Le machine learning a un autre objectif : prévoir.**

## 5. Le machine learning, c'est prévoir

On change **une seule chose** : la cible passe au lendemain. Même outil, même découpage, même code.

### Le pari

On entend souvent que *BTC bouge d'abord, et ETH suit*. Si c'est vrai, le BTC d'aujourd'hui doit prévoir l'ETH de demain.

**Votez à main levée : quel R² sur le test ?** Plutôt 0,7 comme tout à l'heure ? Plutôt 0,3 ? Plutôt 0 ?

### `shift(-1)` : la valeur de demain sur la ligne d'aujourd'hui

`shift(1)`, à la séance 5, faisait descendre une colonne : le prix d'hier sur la ligne d'aujourd'hui. `shift(-1)` la fait **monter** : la valeur de demain sur la ligne d'aujourd'hui.

In [ ]:
rendements["ETH_demain"] = rendements["ETH"].shift(-1)
rendements[["date", "BTC", "ETH", "ETH_demain"]].head()

Lisez la ligne du 11 avril : ETH_demain est l'ETH du 12. Le dernier jour, lui, n'a pas de lendemain : `NaN`, qu'on retire.

On refait **exactement** les gestes des sections 1 à 3.

In [ ]:
demain = rendements.dropna(subset=["ETH_demain"])
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(demain[["BTC"]], demain["ETH_demain"], test_size=0.2, shuffle=False)
modele_d = LinearRegression().fit(X_train_d, y_train_d)

print(r2_score(y_test_d, modele_d.predict(X_test_d)))
print(stats.linregress(X_train_d["BTC"], y_train_d).pvalue)

**R² = −0,005. p-value = 0,22.** Rien. La barre du modèle est même un peu plus longue que celle du modèle bête. Et les deux outils disent la même chose : la statistique, « rien de significatif » ; le test, « rien de prévisible ».

### Écrire

Deuxième essai : le **bitcoin** de demain, à partir du bitcoin d'aujourd'hui. Une colonne `BTC_demain`, puis `r2_btc_demain` (le R² sur le test) et `p_btc_demain` (la p-value sur l'entraînement). Mêmes lignes que la cellule du dessus : copiez, et changez ce qui doit changer.

In [ ]:
_d = rendements.assign(_c=rendements["BTC"].shift(-1)).dropna(subset=["_c"])
_a, _b, _e, _f = train_test_split(_d[["BTC"]], _d["_c"], test_size=0.2, shuffle=False)
verifier("la colonne BTC_demain", "BTC_demain" in rendements.columns and abs(rendements["BTC_demain"][0] - rendements["BTC"][1]) < 1e-9, "shift(-1) sur la colonne BTC")
verifier("r2_btc_demain", abs(r2_btc_demain - r2_score(_f, LinearRegression().fit(_a, _e).predict(_b))) < 1e-6, "r2_score sur le test, avec un modèle ajusté sur l'entraînement")
verifier("p_btc_demain", abs(p_btc_demain - stats.linregress(_a["BTC"], _e).pvalue) < 1e-6, "stats.linregress(...).pvalue, sur l'entraînement")

| | R² sur le test | p-value |
|---|---|---|
| ETH aujourd'hui, à partir de BTC aujourd'hui | **+0,70** | ≈ 0 |
| ETH demain, à partir de BTC aujourd'hui | **−0,005** | 0,22 |
| BTC demain, à partir de BTC aujourd'hui | **−0,015** | 0,11 |

Même outil, mêmes données. Expliquer le présent : 0,70. Prévoir le lendemain : rien.

> **Si ce R² n'était pas nul, on serait riches.** Un modèle d'une ligne, sur des données gratuites, qui prévoit le rendement de demain : on l'appliquerait, on gagnerait, et on ne serait pas dans cette salle. Tout le monde a eu cette idée — et c'est précisément pour ça qu'elle ne marche plus. Dès qu'un rendement devient prévisible, quelqu'un achète, et l'effet disparaît. C'est ce qu'on appelle l'**efficience des marchés**. Vous venez de la tester à la main.

**Une question pour vous.** Un gérant vous présente un modèle avec un R² de 0,85. Quelle est la première question que vous lui posez ?

<details><summary>Réponse</summary>

**Qu'est-ce qu'il prédit, et surtout quand ?** Aujourd'hui à partir d'aujourd'hui, c'est une explication, et 0,85 n'a rien d'extraordinaire. Demain à partir d'aujourd'hui, ce serait une découverte — et il faudrait se demander pourquoi il vous en parle au lieu de s'en servir.
</details>

## 6. Ce qui se prévoit : le volume

Le rendement de demain ne se prévoit pas. **Est-ce que rien ne se prévoit ?**

On change de cible : le **volume**, le montant échangé chaque jour sur le bitcoin, en milliards de dollars.

### Pourquoi quelqu'un voudrait le prévoir

- **Un desk d'exécution** doit vendre 50 millions de dollars de bitcoin cette semaine. Un jour de gros volume, l'ordre se noie dans la masse et fait peu bouger le prix. Un jour creux, il fait baisser le prix contre lui. Choisir le jour, c'est de l'argent.
- **Une plateforme d'échange** vit des frais, donc du volume. Elle doit aussi dimensionner ses serveurs : les jours de rush sont ceux où les plateformes tombent en panne.
- **Un teneur de marché** doit savoir combien de capital mobiliser demain.

### La table

In [ ]:
crypto = pd.read_csv(BASE + "crypto.csv")
crypto["date"] = pd.to_datetime(crypto["date"])
btc = crypto.query("coin == 'BTC'").sort_values("date").reset_index(drop=True)
btc["vol"] = btc["volume"] * btc["close"] / 1e9     # le montant échangé, en milliards de dollars
btc["r"] = btc["close"].pct_change() * 100
btc["vol"].describe()

Un jour ordinaire : 27 milliards de dollars. Les jours de panique : plus de 300.

### Prédire

`autocorr` donne la corrélation d'une colonne avec elle-même, décalée d'un jour : **aujourd'hui ressemble-t-il à hier ?** Pour le rendement, puis pour le volume.

In [ ]:
print(btc["r"].autocorr())

In [ ]:
print(btc["vol"].autocorr())

**−0,05 pour le rendement, +0,78 pour le volume.**

> **Le prix n'a pas de mémoire. Le volume en a.** Un jour agité est suivi d'un jour agité ; un jour de hausse n'est suivi de rien de particulier.

Le machine learning marche là où il y a de la mémoire.

### Deux variables de mémoire : `rolling`

`rolling(5).mean()` : sur chaque ligne, la moyenne **des 5 derniers jours**, aujourd'hui compris. Une moyenne qui avance avec le temps — on dit une moyenne **mobile**. Sur une petite colonne :

In [ ]:
jouet = pd.Series([10.0, 20.0, 30.0, 40.0, 50.0, 60.0, 70.0])
pd.DataFrame({"valeur": jouet, "moyenne sur 3 jours": jouet.rolling(3).mean()})

Les deux premières lignes n'ont pas trois jours derrière elles : `NaN`. La troisième vaut (10 + 20 + 30) / 3 = 20.

### La prévision, dite précisément

> *Chaque soir à minuit, avec ce qu'on sait à la clôture — le volume du jour, sa moyenne sur 5 jours, sa moyenne sur 20 jours — on annonce un nombre : le volume de demain, en milliards de dollars.*

Rien de ce qu'on utilise n'est postérieur à minuit. C'est une vraie prévision.

In [ ]:
btc["m5"] = btc["vol"].rolling(5).mean()
btc["m20"] = btc["vol"].rolling(20).mean()
btc["vol_demain"] = btc["vol"].shift(-1)
v = btc.dropna(subset=["m20", "vol_demain"])

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(v[["vol", "m5", "m20"]], v["vol_demain"], test_size=0.2, shuffle=False)
print(len(X_train_v), len(X_test_v))
print(v["date"][X_test_v.index[0]])

### Retour sur le découpage, parce que c'est maintenant qu'il sert vraiment

Tout à l'heure, le test vérifiait une **relation**. Ici, il **simule l'avenir**. On fait comme si on était le 9 décembre 2024 au soir : on apprend sur tout ce qui précède, et on regarde ce qu'aurait donné le modèle sur les 629 jours suivants, qu'il n'a jamais vus. `shuffle=False` n'est plus une précaution : c'est la condition pour que la simulation soit honnête.

`X` a maintenant **trois colonnes** : le modèle cherchera un coefficient pour chacune. Rien d'autre ne change.

### Écrire

Ajustez `modele_v` sur l'entraînement, puis `prevu_v`, ses prévisions sur le test, et les trois mesures : `mae_v`, `rmse_v`, `r2_v`.

In [ ]:
_p = LinearRegression().fit(X_train_v, y_train_v).predict(X_test_v)
verifier("prevu_v", len(prevu_v) == len(y_test_v) and abs(prevu_v - _p).max() < 1e-6, "fit sur X_train_v, y_train_v ; predict sur X_test_v")
verifier("mae_v", abs(mae_v - mean_absolute_error(y_test_v, _p)) < 1e-6, "mean_absolute_error(y_test_v, prevu_v)")
verifier("rmse_v", abs(rmse_v - mean_squared_error(y_test_v, _p) ** 0.5) < 1e-6, "la racine du mean_squared_error")
verifier("r2_v", abs(r2_v - r2_score(y_test_v, _p)) < 1e-6, "r2_score(y_test_v, prevu_v)")

**R² = 0,42.** Là où le rendement donnait zéro, le volume donne 0,42. Mais contre qui se compare-t-on ?

### Le vrai adversaire : la persistance

Battre la moyenne ne prouve rien. Une prévision doit battre **la règle que n'importe qui applique sans modèle : « demain sera comme aujourd'hui »**. On l'appelle la **persistance**. Sa prévision, c'est simplement la colonne `vol` du test.

### Écrire

`prevu_persistance`, puis ses trois mesures : `mae_p`, `rmse_p`, `r2_p`.

In [ ]:
verifier("mae_p", abs(mae_p - mean_absolute_error(y_test_v, X_test_v["vol"])) < 1e-6, "la prévision de la persistance est X_test_v['vol']")
verifier("rmse_p", abs(rmse_p - mean_squared_error(y_test_v, X_test_v["vol"]) ** 0.5) < 1e-6, "la racine du mean_squared_error")
verifier("r2_p", abs(r2_p - r2_score(y_test_v, X_test_v["vol"])) < 1e-6, "r2_score(y_test_v, prevu_persistance)")

Et le modèle bête, pour compléter le tableau : toujours la moyenne de l'entraînement.

In [ ]:
prevu_bete = np.full(len(y_test_v), y_train_v.mean())
comparaison = pd.DataFrame({
    "MAE (Md$)": [mean_absolute_error(y_test_v, prevu_bete), mae_p, mae_v],
    "RMSE (Md$)": [mean_squared_error(y_test_v, prevu_bete) ** 0.5, rmse_p, rmse_v],
    "R²": [r2_score(y_test_v, prevu_bete), r2_p, r2_v],
}, index=["modèle bête : la moyenne", "persistance : demain = aujourd'hui", "régression"])
comparaison.round(2)

**La régression bat les deux, sur les trois mesures.**

- **MAE : 12,2 milliards**, pour un volume moyen de 46 sur la période : on se trompe d'environ un quart.
- **RMSE / MAE = 1,4** : quelques journées folles pèsent lourd.
- **R² : 0,42** contre 0,26 pour la persistance. Et c'est le R², sans unité, qui permet de dire : le volume se prévoit, le rendement (−0,005) non.

Le modèle bête a un R² de −0,74 : la moyenne de 2018-2024 est beaucoup trop basse pour 2025-2026. C'est la barre négative annoncée à la section 3.

### Ce que le modèle a appris

In [ ]:
print(modele_v.coef_)

Trois coefficients, un par colonne : **0,35** pour le volume du jour, **0,22** pour la semaine, **0,39** pour le mois. Le modèle n'a pas appris « demain = aujourd'hui » : il a appris **comment** ça continue. Un jour exceptionnel se dégonfle vers la moyenne du mois ; une tendance de fond s'installe.

### Bonus, si le temps le permet

Changez `MONNAIE` et relancez. Attendus sur le test : ETH 0,55, SOL 0,55, DOGE 0,60, BNB 0,72. Le bitcoin, le plus surveillé des sept, est le moins prévisible.

In [ ]:
MONNAIE = "ETH"

m = crypto.query("coin == @MONNAIE").sort_values("date").reset_index(drop=True)
m["vol"] = m["volume"] * m["close"] / 1e9
m["m5"] = m["vol"].rolling(5).mean()
m["m20"] = m["vol"].rolling(20).mean()
m["vol_demain"] = m["vol"].shift(-1)
m = m.dropna(subset=["m20", "vol_demain"])
a, b, c, d = train_test_split(m[["vol", "m5", "m20"]], m["vol_demain"], test_size=0.2, shuffle=False)
print(MONNAIE, r2_score(d, LinearRegression().fit(a, c).predict(b)), r2_score(d, b["vol"]))

## 7. Entraînement contre test : le par-cœur

### Écrire

Les trois mesures du modèle de volume, cette fois **sur l'entraînement** : `mae_train`, `rmse_train`, `r2_train`.

In [ ]:
_q = modele_v.predict(X_train_v)
verifier("mae_train", abs(mae_train - mean_absolute_error(y_train_v, _q)) < 1e-6, "les prévisions sur X_train_v, comparées à y_train_v")
verifier("rmse_train", abs(rmse_train - mean_squared_error(y_train_v, _q) ** 0.5) < 1e-6, "la racine du mean_squared_error")
verifier("r2_train", abs(r2_train - r2_score(y_train_v, _q)) < 1e-6, "r2_score(y_train_v, ...)")

In [ ]:
pd.DataFrame({"MAE (Md$)": [mae_train, mae_v], "RMSE (Md$)": [rmse_train, rmse_v], "R²": [r2_train, r2_v]},
             index=["entraînement", "test"]).round(2)

**Le modèle est meilleur sur l'entraînement, sur les trois mesures. C'est normal, et il faut comprendre pourquoi.**

La droite a été choisie pour coller à **ces** jours-là : c'est la définition des moindres carrés. Elle a vu les corrigés. Sur le test, elle découvre les questions. Le chiffre qui compte, c'est **toujours** celui du test ; celui de l'entraînement est une borne optimiste.

> Attention à la MAE : 5,6 contre 12,2 milliards, ça a l'air d'un gouffre. Mais le volume moyen a presque doublé entre les deux périodes (27 contre 46 milliards). Rapportée au volume, l'erreur passe de 21 % à 26 %. **On ne compare deux MAE que sur la même échelle** — c'est le travail du R².

### Poussé jusqu'au bout : apprendre par cœur

Retour au rendement d'ETH de demain, celui qui ne se prévoit pas. Et si on donnait au modèle **beaucoup** de variables ? Les rendements des sept monnaies aujourd'hui, hier, avant-hier… jusqu'à 40 jours en arrière : 280 colonnes.

La cellule suivante le fait pour 7, 35, 70, 140 et 280 variables. Elle est longue ; **vous n'avez pas à l'écrire**, seulement à la lire en diagonale et à l'exécuter.

In [ ]:
monnaies = ["BTC", "ETH", "SOL", "DOGE", "XRP", "BNB", "ADA"]
lignes = []
for jours in [1, 5, 10, 20, 40]:
    colonnes = {}
    for monnaie in monnaies:
        for k in range(jours):
            colonnes[monnaie + "_il_y_a_" + str(k)] = rendements[monnaie].shift(k)   # le rendement d'il y a k jours
    table = pd.DataFrame(colonnes)
    table["cible"] = rendements["ETH"].shift(-1)
    table = table.dropna()
    a, b, c, d = train_test_split(table.drop(columns="cible"), table["cible"], test_size=0.2, shuffle=False)
    mod = LinearRegression().fit(a, c)
    lignes.append([len(a.columns), r2_score(c, mod.predict(a)), r2_score(d, mod.predict(b))])

par_coeur = pd.DataFrame(lignes, columns=["variables", "R² entraînement", "R² test"])
par_coeur.round(3)

Plus on ajoute de variables, plus le modèle « explique » l'entraînement — **21 % avec 280 variables, sur une cible imprévisible !** — et plus il s'effondre sur le test.

Il n'a rien compris : il a **appris le bruit par cœur**. Avec 280 boutons à régler, il trouve toujours une combinaison qui colle au passé, et qui ne veut rien dire pour l'avenir. C'est l'étudiant qui récite les corrigés.

> **Un grand écart entre l'entraînement et le test, c'est le signe du par-cœur.** On l'appelle le **surapprentissage**. C'est l'erreur la plus fréquente du machine learning, et la raison d'être du découpage.

### Une question pour les attentifs

Sur ETH et BTC du même jour, à la section 4, le R² de l'entraînement valait 0,64 et celui du test 0,70. **Le test est meilleur que l'entraînement. Comment est-ce possible ?**

<details><summary>Réponse</summary>

« Meilleur sur l'entraînement », c'est vrai **à période égale**. Ici, le test est une autre période : en 2025-2026, ETH a suivi BTC de plus près qu'avant. Le test n'est pas une copie de l'entraînement, c'est un autre moment du marché — et c'est justement ce qu'on veut mesurer. Le par-cœur, lui, se voit quand l'écart est grand et qu'il grandit avec le nombre de variables.
</details>

## 8. Ce que vous savez faire

| Vous voulez... | Vous écrivez |
|---|---|
| mettre des données de côté, sans mélanger les dates | `train_test_split(X, y, test_size=0.2, shuffle=False)` |
| ajuster une droite | `modele = LinearRegression().fit(X_train, y_train)` |
| lire ses coefficients | `modele.coef_`, `modele.intercept_` |
| prévoir | `modele.predict(X_test)` |
| le R², la MAE, le RMSE | `r2_score(y, prevu)`, `mean_absolute_error(y, prevu)`, `mean_squared_error(y, prevu) ** 0.5` |
| la pente, son erreur-type, sa p-value | `stats.linregress(x, y)` |
| la valeur de demain sur la ligne d'aujourd'hui | `col.shift(-1)` |
| une moyenne mobile | `col.rolling(5).mean()` |
| la mémoire d'une colonne | `col.autocorr()` |

## Cinq phrases à retenir

1. **Un modèle se note sur des données qu'il n'a jamais vues.** L'examen, pas le problem set. Sur des dates, `shuffle=False`.
2. **Le R² est la part de l'erreur du modèle bête qu'on fait disparaître.** La MAE dit de combien on se trompe ; le RMSE, si de grosses erreurs se cachent.
3. **Expliquer n'est pas prévoir.** La même droite vaut 0,70 sur le présent et 0 sur le lendemain.
4. **Le prix n'a pas de mémoire, le volume en a.** Et une prévision se juge contre la persistance, pas seulement contre la moyenne.
5. **Meilleur à l'entraînement qu'au test, c'est normal. Beaucoup meilleur, c'est du par-cœur.**

## Pour la prochaine séance

La prochaine séance garde le volume du bitcoin, mais change la question : non plus **combien** demain, mais **oui ou non** — demain sera-t-il une journée de gros volume ?

Trois cellules à prédire, une à écrire.

In [ ]:
print(r2_score(y_test_v, y_test_v))

In [ ]:
print(r2_score(y_test_v, np.full(len(y_test_v), y_test_v.mean())))

In [ ]:
print(mean_absolute_error(y_test_v, y_test_v + 1))

Le volume **d'ETH** de demain, à partir du volume d'ETH du jour **seulement** (une variable), dans `r2_eth_seul`. Fait-il mieux ou moins bien que les trois variables du bonus ?

In [ ]:
_e = crypto.query("coin == 'ETH'").sort_values("date").reset_index(drop=True)
_e["v"] = _e["volume"] * _e["close"] / 1e9
_e["d"] = _e["v"].shift(-1)
_e = _e.dropna(subset=["d"])
_a, _b, _c, _d = train_test_split(_e[["v"]], _e["d"], test_size=0.2, shuffle=False)
verifier("r2_eth_seul", abs(r2_eth_seul - r2_score(_d, LinearRegression().fit(_a, _c).predict(_b))) < 0.01, "une seule colonne dans X : e[['vol']], et la cible shift(-1)")